<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-10-tuning-and-evaluation/lesson-10.4-evaluation/notebooks/GCP_Capstone_10.4_Evaluation.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 10.4 Vertex AI Evaluation — ROUGE/BLEU, Gemini-as-Judge, Trajectory, Experiments
**Netsetos GenAI Engineering — GCP Capstone**


In [ ]:
!pip install -q google-cloud-aiplatform[evaluation] pandas

import pandas as pd
from vertexai.evaluation import (
    EvalTask, PointwiseMetric, PairwiseMetric,
    PointwiseMetricPromptTemplate, MetricPromptTemplateExamples
)
import vertexai

# Application Default Credentials on Colab (no API keys). No-op off Colab.
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    pass

PROJECT = 'documind-ai-YOUR-ID'
LOCATION = 'us-central1'

vertexai.init(project=PROJECT, location=LOCATION, experiment='documind-eval-demo')
print('SDK ready')


## Cell 1: Build Evaluation Dataset


In [ ]:
# DocuMind document classification eval dataset
eval_df = pd.DataFrame({
    'prompt': [
        'Classify this document (INVOICE/CONTRACT/REPORT/RECEIPT):\nInvoice #INV-2024-001, Amount $4,590.00, Due 2024-04-14',
        'Classify this document:\nService Agreement between ACME and ClientX, Term 24 months effective 2024-01-01',
        'Classify this document:\nQ3 2024 Financial Report - Revenue grew 18% YoY to $47.2M',
        'Classify this document:\nReceipt #R-5521 from OfficeSupplies Inc, Date 2024-03-20, Total $114.50',
        'Classify this document:\nInvoice #INV-2024-002 for Cloud Services, $12,000, Net 30',
    ],
    'reference': ['INVOICE', 'CONTRACT', 'REPORT', 'RECEIPT', 'INVOICE'],
    # Pre-generated responses to avoid live inference during demo
    'response': ['INVOICE', 'CONTRACT', 'REPORT', 'RECEIPT', 'INVOICE'],
})

print(f'Dataset: {len(eval_df)} rows')
print(eval_df.head())


## Cell 2: Computation-Based Metrics (exact_match, ROUGE)


In [ ]:
# Evaluate pre-generated responses with computation metrics
eval_task_classification = EvalTask(
    dataset=eval_df,
    metrics=['exact_match', 'rouge_l_sum', 'bleu'],
    experiment='documind-classification-comp',
)

# Template (requires Vertex AI Experiments to be initialized)
print('Computation metrics configured:')
print('  exact_match   - binary string match')
print('  rouge_l_sum   - LCS F1 at summary level')
print('  bleu          - n-gram precision with brevity penalty')
print()
print('Template call (requires live auth):')
print('  result = eval_task_classification.evaluate(')
print('      model="gemini-3.6-flash",')
print('      experiment_run_name="flash-v1")')
print('  result.summary_metrics  # aggregated scores')
print('  result.metrics_table    # per-row DataFrame')


## Cell 3: Gemini-as-Judge Pointwise Metrics


In [ ]:
# RAG answer quality eval dataset
rag_df = pd.DataFrame({
    'prompt': [
        'What are the payment terms for invoice INV-2024-001?',
        'What is the contract term?',
    ],
    'context': [
        'Invoice #INV-2024-001, Amount $4,590.00, Payment Terms: Net 30, Due 2024-04-14',
        'Service Agreement. Term: 24 months effective 2024-01-01.',
    ],
    'response': [
        'Payment terms are Net 30, with the invoice due on April 14, 2024.',
        'The contract term is 24 months, effective January 1, 2024.',
    ],
    'instruction': [
        'Answer the question using only the provided context.',
        'Answer the question using only the provided context.',
    ],
})

eval_task_rag = EvalTask(
    dataset=rag_df,
    metrics=[
        MetricPromptTemplateExamples.Pointwise.GROUNDEDNESS,
        MetricPromptTemplateExamples.Pointwise.FLUENCY,
        MetricPromptTemplateExamples.Pointwise.INSTRUCTION_FOLLOWING,
        MetricPromptTemplateExamples.Pointwise.QUESTION_ANSWERING_QUALITY,
    ],
    experiment='documind-rag-quality',
)

print('Pointwise Gemini-as-judge metrics configured:')
print('  groundedness                 - factual consistency with context')
print('  fluency                      - grammar, naturalness')
print('  instruction_following        - adherence to constraints')
print('  question_answering_quality   - QA response quality')
print()
print('Each returns integer 1-5 + chain-of-thought explanation')


## Cell 4: Custom PointwiseMetric with Rubric


In [ ]:
# DocuMind invoice extraction custom metric
extraction_accuracy = PointwiseMetric(
    metric='invoice_extraction_accuracy',
    metric_prompt_template=PointwiseMetricPromptTemplate(
        criteria={
            'Field Completeness': 'Does the extraction capture ALL required fields (invoice_id, vendor, date, total, line_items)?',
            'Value Accuracy':     'Are extracted values correct when compared against the source document?',
            'Format Compliance':  'Do extracted values use expected formats? Dates in ISO-8601 (YYYY-MM-DD), amounts as numbers (no currency symbols), line items as arrays.',
        },
        rating_rubric={
            '5': 'All fields correct, complete, properly formatted. Production-ready.',
            '4': 'Minor formatting issues (e.g., date format), all fields present and accurate.',
            '3': 'One field missing or one value incorrect.',
            '2': 'Multiple fields missing or multiple values incorrect.',
            '1': 'Extraction largely failed - most fields missing or wrong.',
        },
        input_variables=['prompt', 'response', 'reference'],
    ),
)

print('Custom metric: invoice_extraction_accuracy')
print(f'  3 criteria: {list(extraction_accuracy.metric_prompt_template.criteria.keys())}')
print(f'  5-point rubric with explicit pass/fail guidance')
print()
print('Usage:')
print('  task = EvalTask(dataset=extraction_df, metrics=[extraction_accuracy])')
print('  result = task.evaluate(model=model)')


## Cell 5: Pairwise Comparison (LoRA vs Base)


In [ ]:
# Compare LoRA-tuned DocuMind model against base Gemini Flash
# (template - requires tuned endpoint from Lesson 10.1)

def build_pairwise_task(eval_df, baseline_model_name='gemini-3.6-flash'):
    '''Returns EvalTask configured for pairwise comparison.'''
    pairwise_qa = PairwiseMetric(
        metric='pairwise_question_answering_quality',
        metric_prompt_template=MetricPromptTemplateExamples.get_prompt_template(
            'pairwise_question_answering_quality'),
        baseline_model=baseline_model_name,
    )
    
    return EvalTask(
        dataset=eval_df,
        metrics=[
            pairwise_qa,
            'pairwise_groundedness',
            'pairwise_instruction_following',
        ],
        experiment='documind-lora-vs-base',
    )

pairwise_task = build_pairwise_task(rag_df)

print('Pairwise task configured:')
print('  Baseline: gemini-3.6-flash (base model)')
print('  Candidate: LoRA-tuned endpoint')
print('  Metrics: pairwise QA quality, groundedness, instruction following')
print()
print('Summary output structure:')
print('  candidate_model_win_rate  - proportion 0.0-1.0')
print('  baseline_model_win_rate')
print('  tie_rate')
print()
print('Min sample size: 400 examples for statistical confidence')


## Cell 6: Agent Trajectory Metrics


In [ ]:
# DocuMind agent tool-calling evaluation
agent_df = pd.DataFrame({
    'prompt': [
        'Find invoices from Vendor X and summarize payment terms',
        'Extract vendor and total from the latest invoice',
    ],
    'reference_trajectory': [
        [
            {'tool_name': 'search_documents', 'tool_input': {'vendor': 'X', 'doc_type': 'INVOICE'}},
            {'tool_name': 'extract_fields', 'tool_input': {'fields': ['payment_terms']}},
            {'tool_name': 'summarize', 'tool_input': {'max_sentences': 3}},
        ],
        [
            {'tool_name': 'search_documents', 'tool_input': {'doc_type': 'INVOICE', 'limit': 1}},
            {'tool_name': 'extract_fields', 'tool_input': {'fields': ['vendor', 'total']}},
        ],
    ],
    'predicted_trajectory': [
        [
            {'tool_name': 'search_documents', 'tool_input': {'vendor': 'X', 'doc_type': 'INVOICE'}},
            {'tool_name': 'extract_fields', 'tool_input': {'fields': ['payment_terms']}},
            {'tool_name': 'summarize', 'tool_input': {'max_sentences': 3}},
        ],
        [
            {'tool_name': 'search_documents', 'tool_input': {'doc_type': 'INVOICE'}},
            {'tool_name': 'extract_fields', 'tool_input': {'fields': ['vendor', 'total', 'date']}},  # extra field
        ],
    ],
})

# Compute trajectory metrics programmatically (demo)
def trajectory_exact_match(pred, ref):
    return int(pred == ref)

def trajectory_in_order_match(pred, ref):
    pred_names = [t['tool_name'] for t in pred]
    ref_names = [t['tool_name'] for t in ref]
    i = 0
    for name in pred_names:
        if i < len(ref_names) and name == ref_names[i]:
            i += 1
    return int(i == len(ref_names))

def trajectory_any_order_match(pred, ref):
    pred_names = set(t['tool_name'] for t in pred)
    ref_names = set(t['tool_name'] for t in ref)
    return int(ref_names.issubset(pred_names))

def trajectory_precision(pred, ref):
    if not pred: return 0.0
    ref_names = set(t['tool_name'] for t in ref)
    relevant = sum(1 for t in pred if t['tool_name'] in ref_names)
    return relevant / len(pred)

def trajectory_recall(pred, ref):
    if not ref: return 0.0
    pred_names = set(t['tool_name'] for t in pred)
    captured = sum(1 for t in ref if t['tool_name'] in pred_names)
    return captured / len(ref)

print('Trajectory metrics on 2 DocuMind examples:')
print(f'{"Metric":<30} {"Row 1":>7} {"Row 2":>7}')
print('-' * 50)
for name, fn in [
    ('trajectory_exact_match', trajectory_exact_match),
    ('trajectory_in_order_match', trajectory_in_order_match),
    ('trajectory_any_order_match', trajectory_any_order_match),
    ('trajectory_precision', trajectory_precision),
    ('trajectory_recall', trajectory_recall),
]:
    r1 = fn(agent_df['predicted_trajectory'][0], agent_df['reference_trajectory'][0])
    r2 = fn(agent_df['predicted_trajectory'][1], agent_df['reference_trajectory'][1])
    print(f'{name:<30} {r1:>7.2f} {r2:>7.2f}')


## Cell 7: Vertex AI Experiments Tracking


In [ ]:
# Systematic prompt comparison with Experiments integration
prompt_templates = [
    'Classify the document (INVOICE/CONTRACT/REPORT/RECEIPT): {prompt}',
    'You are DocuMind. Return the category name only. Document: {prompt}',
    'Task: classify. Options: INVOICE, CONTRACT, REPORT, RECEIPT.\nDocument: {prompt}\nCategory:',
]

eval_task_prompts = EvalTask(
    dataset=eval_df,
    metrics=['exact_match', 'rouge_l_sum'],
    experiment='documind-prompt-comparison',
)

print('Prompt comparison pattern:')
print()
print('for idx, template in enumerate(prompt_templates):')
print('    result = eval_task_prompts.evaluate(')
print('        model="gemini-3.6-flash",')
print('        prompt_template=template,')
print('        experiment_run_name=f"prompt-{idx}")')
print()
print('All runs land in experiment "documind-prompt-comparison"')
print('View in Console: Vertex AI -> Model Development -> Experiments')
print()
print('Visualize in Colab:')
print('from vertexai.preview.evaluation import notebook_utils')
print('notebook_utils.display_radar_plot([("p0", r0), ("p1", r1), ("p2", r2)],')
print('    metrics=["exact_match", "rouge_l_sum"])')


## Cell 8: Production Evaluation Pipeline


In [ ]:
# DocuMind end-to-end evaluation strategy
class DocuMindEvaluator:
    '''Continuous evaluation across dev, pre-prod, production.'''
    
    def __init__(self, experiment_name: str):
        self.experiment = experiment_name
    
    def dev_ci_gate(self, eval_df, model):
        '''CI/CD quality gate. Block merge if scores below threshold.'''
        task = EvalTask(
            dataset=eval_df,
            metrics=['exact_match', 'rouge_l_sum',
                     MetricPromptTemplateExamples.Pointwise.GROUNDEDNESS],
            experiment=self.experiment,
        )
        result = task.evaluate(model=model, experiment_run_name=f'ci-{_current_commit()}')
        
        THRESHOLDS = {
            'exact_match/mean': 0.85,
            'groundedness/mean': 4.0,
        }
        for metric, threshold in THRESHOLDS.items():
            if result.summary_metrics.get(metric, 0) < threshold:
                raise AssertionError(f'{metric} = {result.summary_metrics[metric]} < {threshold}')
        return result
    
    def preprod_pairwise(self, eval_df, candidate_model, baseline_model):
        '''Pre-production A/B. Require win_rate > 55% to ship.'''
        pairwise = PairwiseMetric(
            metric='pairwise_question_answering_quality',
            metric_prompt_template=MetricPromptTemplateExamples.get_prompt_template(
                'pairwise_question_answering_quality'),
            baseline_model=baseline_model,
        )
        task = EvalTask(dataset=eval_df, metrics=[pairwise], experiment=self.experiment)
        result = task.evaluate(model=candidate_model,
                                experiment_run_name='preprod-pairwise')
        
        win_rate = result.summary_metrics.get('pairwise_question_answering_quality/candidate_model_win_rate', 0)
        if win_rate < 0.55:
            raise AssertionError(f'win_rate {win_rate} < 0.55 - do not ship')
        return result
    
    def production_shadow_canary(self, sample_df):
        '''Reference-free production monitoring on 1% of traffic.'''
        task = EvalTask(
            dataset=sample_df,  # response column pre-populated from canary
            metrics=[
                MetricPromptTemplateExamples.Pointwise.GROUNDEDNESS,
                MetricPromptTemplateExamples.Pointwise.SAFETY,
            ],
            experiment=self.experiment,
        )
        result = task.evaluate(experiment_run_name='canary-daily')
        return result.summary_metrics  # Alert if drops > 10%

def _current_commit():
    return 'abc1234'  # Placeholder for git rev-parse

print('DocuMindEvaluator ready')
print('Three-tier strategy: dev_ci_gate -> preprod_pairwise -> production_shadow_canary')
print('All runs tracked in Vertex AI Experiments for auditable model evolution')
print()
print('Cost profile:')
print('  CI gate:         ~$0.075/run (400 examples x 3 metrics)')
print('  Pre-prod:        ~$0.50/run (400 examples, pairwise)')
print('  Production:      ~$5/day (1% traffic sampling)')


## Done!
- Computation metrics (exact_match, BLEU, ROUGE-L) with reference data
- Gemini-as-judge pointwise (groundedness, fluency, instruction_following)
- Custom PointwiseMetric with criteria + 5-point rubric
- Pairwise metrics for A/B (LoRA vs base)
- Trajectory metrics (5 variants) for agent tool-calling
- Vertex AI Experiments integration via experiment + experiment_run_name
- Production 3-tier pipeline: dev CI gate, pre-prod pairwise, production canary
